In [1]:
!nvidia-smi

Fri Dec 26 23:56:29 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      On  |   00000000:00:03.0 Off |                    0 |
| N/A   77C    P0             34W /   72W |   13750MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[0]
sys.path.append(str(project_root))

In [3]:
from modules.lizard import create_lizard_attention_config

In [4]:
import torch

In [5]:
cfg = create_lizard_attention_config(
    batch_size=2,
    seq_len=8192 * 4,
    hidden_size=256,
    num_heads=4,
    dtype="float16",
    device="cuda" if torch.cuda.is_available() else "cpu",
)

In [6]:
from modules.lizard_attention_block_pytorch import LizardAttentionBlock

In [7]:
torch_dtype = cfg.dtype.to_torch()
head_dim = cfg.hidden_size // cfg.num_heads

print(f"Configuring model with: {cfg}")
print(f"Head Dimension: {head_dim}")

model = (
    LizardAttentionBlock(
        d_model=cfg.hidden_size, n_heads=cfg.num_heads, window_size=32, alpha=0.5, m=4
    )
    .to(cfg.device)
    .to(torch_dtype)
)

q = torch.randn(
    cfg.batch_size,
    cfg.num_heads,
    cfg.seq_len,
    head_dim,
    device=cfg.device,
    dtype=torch_dtype,
)
k = torch.randn(
    cfg.batch_size,
    cfg.num_heads,
    cfg.seq_len,
    head_dim,
    device=cfg.device,
    dtype=torch_dtype,
)
v = torch.randn(
    cfg.batch_size,
    cfg.num_heads,
    cfg.seq_len,
    head_dim,
    device=cfg.device,
    dtype=torch_dtype,
)

output = model(q, k, v)

print(f"Input shape:  {q.shape}")
print(f"Output shape: {output.shape}")

assert q.shape == output.shape

Configuring model with: LizardAttentionBlockConfig(batch_size=2, seq_len=32768, hidden_size=256, num_heads=4, dtype=DTypeConfig(name='float16'), device='cuda')
Head Dimension: 64
Input shape:  torch.Size([2, 4, 32768, 64])
Output shape: torch.Size([2, 4, 32768, 64])


In [ ]:
# 1. Warmup (Crucial!)
# Run the model a few times to compile kernels and populate caches.
# If you skip this, your first measurement will be 10x-100x slower.
print("Warming up...")
for _ in range(10):
    _ = model(q, k, v)
torch.cuda.synchronize()

# 2. Benchmark Loop
start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)
timings = []

print("Benchmarking...")
with torch.no_grad(): # Disable gradients for pure inference speed
    for _ in range(100): # Run 100 times for statistical stability
        start_event.record()
        _ = model(q, k, v)
        end_event.record()
        
        # Wait for this specific run to finish
        torch.cuda.synchronize()
        timings.append(start_event.elapsed_time(end_event))

# 3. Report
import numpy as np
mean_time = np.mean(timings)
std_time = np.std(timings)

print(f"Average latency: {mean_time:.3f} ms ± {std_time:.3f} ms")

Warming up...
Benchmarking...
Average latency: 62.708 ms ± 0.481 ms


: 